In [14]:
from Functions_wrapped import *
import matplotlib.pyplot as plt
from scipy import optimize as opt
from scipy import stats as sts
from BG2_functions import *
import time


In [15]:
ls_ps=[0.2,0.1,0.05,0.02,0.01,0.005,0.002,0.001,0.0005,0.0002,0.0001]
ls_N=[]
ls_D=[]
rows=[]
for pi in ls_ps:
    bbg2= better_BG2(pi)
    EGS=bbg2[0]
    rows.append({'Prevalence': pi, 'EGS': EGS})
    new_N=EGS*10
    new_D=np.ceil(new_N*pi)
    new_n=np.round(new_D/pi)
    ls_N.append(int(new_n))
    ls_D.append(int(new_D))
bg2_df = pd.DataFrame(rows)

In [ ]:

M_id=12
# ── User-defined parameters ──────────────────────────────────────────────────
N_list    =ls_N[:M_id]# [500, 200, 500, 1000]   # sample sizes to sweep
diff_list =ls_D[:M_id]# [25,   2,  1,   1]   # differentiation level for each N (same length as N_list)

checks_hierarchical = 500        # MC checks for hierarchical simulation
metrics_checks      = 1e0        # max_checks passed to mean_metrics_fast (internal scaler=1e3)

# Random design parameters (from rand_WA_wrapped.py defaults)
rand_guesses          = 2    # number of random WA candidates to evaluate
rand_max_redundancy   = 2.0   # upper bound on well-redundancy relative to N*log2(N)
rand_min_redundancy   = 0.5   # lower bound on well-redundancy
rand_n_compounds_per_well = 0 # 0 = auto-select
rand_n_wells          = 0     # 0 = auto-select
rand_max_compounds    = 0     # 0 = auto-select (uses get_max_C default)
# ────────────────────────────────────────────────────────────────────────────

rows = []

for N, diff in zip(N_list, diff_list):
    # Maximum number of multidim dimensions that make sense for this N
    # (require L1 = ceil(N^(1/k)) >= 2  =>  k <= log2(N))
    max_dims = max(3, int(np.floor(np.log2(max(N, 2)))))

    # ── Build list of (method_name, assign_fn, kwargs) for deterministic methods
    methods_spec = []

    # Matrix (multidim-2 alias)
    methods_spec.append(('Matrix', assign_wells_mat, {'n_compounds': N}))

    # Multidim 3 … max_dims
    nd=3
    methods_spec.append((f'multidim-{nd}', assign_wells_multidim,
                            {'n_compounds': N, 'n_dims': nd}))

    # Binary
    methods_spec.append(('Binary', assign_wells_bin,
                         {'n_compounds': N, 'differentiate': diff}))


    # ── Deterministic methods: time WA construction + metrics calculation ─────
    for method_name, wa_fn, wa_kwargs in methods_spec:
        t0 = time.perf_counter()
        WA = wa_fn(**wa_kwargs)
        result = mean_metrics_fast(well_assigner=WA, differentiate=diff,
                                   max_checks=metrics_checks)
        elapsed = time.perf_counter() - t0
        rows.append({'N': N, 'diff': diff, 'Method': method_name,'Prevalence': diff/N,
                     'time': elapsed, 'mean_tests': result[0]})

    # ── Random: assign_wells_random_precomp already returns mean_tests ────────
    if False:
        t0 = time.perf_counter()
        _, rand_mean_tests, _ = assign_wells_random_precomp(
            n_compounds=N,
            differentiate=diff,
            guesses=rand_guesses,
            max_redundancy=rand_max_redundancy,
            min_redundancy=rand_min_redundancy,
            n_compounds_per_well=rand_n_compounds_per_well,
            n_wells=rand_n_wells,
            max_compounds=rand_max_compounds,
            return_me=True,
        )
        elapsed = time.perf_counter() - t0
        rows.append({'N': N, 'diff': diff, 'Method': 'Random',
                    'time': elapsed, 'mean_tests': rand_mean_tests})

    # ── Hierarchical: calculate_full_metrics_hierarchical_fast ────────────────
    t0 = time.perf_counter()
    hier = calculate_metrics_hierarchical_fast(N, diff, checks=checks_hierarchical, 
                                               keep_ratios_constant=True, Faster=True)
    elapsed = time.perf_counter() - t0
    # hier[0] = mean total experiments
    rows.append({'N': N, 'diff': diff, 'Method': 'Hierarchical','Prevalence': diff/N, 
                 'time': elapsed, 'mean_tests': hier[0]})

timing_df = pd.DataFrame(rows, columns=['N', 'diff', 'Prevalence', 'Method', 'time', 'mean_tests'])
timing_df['ET'] = timing_df['mean_tests'] / timing_df['N']
timing_df[['N', 'diff','Method', 'Prevalence',  'ET', 'time']]



In [ ]:
timing_df.to_csv("timing_results_PoolPy.csv", index=False)


In [16]:
rows = []

for p in ls_ps:
    t0 = time.perf_counter()
    #DM=int(np.max([np.log2(1/p),9])+1)
    DM=50
    result=brute_better_BG2(p,DM)
    elapsed = time.perf_counter() - t0
    rows.append({'Method': 'BG_2','Prevalence': p,
                    'time': elapsed, 'mean_tests': result[-1], 'size': len(result[0])})
    t0 = time.perf_counter()
    result=brute_BG2_Gen_MD(p,DM)
    elapsed = time.perf_counter() - t0
    rows.append({'Method': 'BG_2_MD','Prevalence': p,
                    'time': elapsed, 'mean_tests': result[-1], 'size': result[0]})

timing_df = pd.DataFrame(rows, columns=['Prevalence', 'Method', 'time', 'mean_tests', 'size'])
timing_df['ET'] = timing_df['mean_tests']
timing_df[['Method', 'Prevalence', 'size',  'ET', 'time']]



/Users/ltalamanca/My Drive/Git/PoolPy/BG2_functions.py:135: RuntimeWarning: divide by zero encountered in scalar divide
  return((D*N+np.sum(ev))/(N**D))


,Method,Prevalence,size,ET,time
0,BG_2,0.2000,1,0.821333,0.021018
1,BG_2_MD,0.2000,2,0.889244,0.037431
2,BG_2,0.1000,2,0.586304,0.020342
3,BG_2_MD,0.1000,3,0.670098,0.026347
4,BG_2,0.0500,2,0.376986,0.004291
5,BG_2_MD,0.0500,2,0.426823,0.025999
6,BG_2,0.0200,3,0.197977,0.006028
7,BG_2_MD,0.0200,2,0.249976,0.027431
8,BG_2,0.0100,4,0.117908,0.007618
9,BG_2_MD,0.0100,3,0.167767,0.029392


In [4]:
timing_df.to_csv("timing_results_our_BG2.csv", index=False)